# 🚀 YOLOv11 RDD2022 Crack Training Notebook (v3)
## 🔬 Specialized Model for Dashboard-View Crack Detection

---

**Goal:** Train a high-accuracy (>75% mAP) model for detecting Longitudinal (D00) and Transverse (D10) cracks.

**Filters Applied:**
- **Classes:** D00 (Longitudinal) + D10 (Transverse) only
- **Countries:** India, Japan, Czech Republic only (smartphone data)

**Steps:**
1. Enable GPU runtime
2. Install dependencies
3. Download & Filter (Class + Country)
4. Configure & Train
5. Download Model

## 1️⃣ Enable GPU Runtime
**Go to: Runtime → Change runtime type → T4 GPU**

In [ ]:
# Check GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → GPU")

## 2️⃣ Install Dependencies

In [ ]:
# Install Ultralytics and Kaggle CLI (pinned version)
!pip uninstall -y kaggle kagglesdk -q
!pip install -q kaggle==1.5.16
!pip install -q -U ultralytics
print("✅ Dependencies installed!")

## 3️⃣ Download & Filter RDD2022

In [ ]:
import os
import json

# ===========================
# 1. SETUP KAGGLE AUTH
# ===========================
KAGGLE_USERNAME = "" # Enter your Kaggle username here
KAGGLE_KEY = ""      # Enter your Kaggle API key here

if KAGGLE_USERNAME and KAGGLE_KEY:
    kaggle_creds = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
    with open('kaggle.json', 'w') as f:
        json.dump(kaggle_creds, f)

if not os.path.exists("kaggle.json"):
    print("❌ kaggle.json not found! Upload it or fill in credentials above.")
else:
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✅ Kaggle Auth Configured")

    # ===========================
    # 2. DOWNLOAD DATASET
    # ===========================
    if not os.path.exists("rdd2022"):
        print("⬇️ Downloading RDD2022...")
        !kaggle datasets download -d aliabdelmenam/rdd-2022
        print("📦 Unzipping...")
        !unzip -q rdd-2022.zip -d rdd2022
        print("✅ Download & Unzip Complete!")
    else:
        print("✅ RDD2022 already downloaded.")

In [ ]:
# ===========================
# 3. SMART FILTER (Class + Country)
# ===========================
import shutil
from tqdm import tqdm
import os

# Configuration
SOURCE_ROOT = "/content/rdd2022/RDD_SPLIT"
DEST_DIR = "/content/rdd_cracks_only"
KEEP_CLASSES = [0, 1]  # 0=D00 (Longitudinal), 1=D10 (Transverse)
KEEP_COUNTRIES = ["Japan", "India", "Czech"]  # Smartphone data only

def is_allowed_country(filename):
    """Check if filename starts with an allowed country prefix."""
    for country in KEEP_COUNTRIES:
        if filename.startswith(country):
            return True
    return False

if os.path.exists(DEST_DIR):
    print("✅ Filtered dataset already exists. Delete folder to re-filter.")
else:
    # Setup Directories
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{DEST_DIR}/images/{split}", exist_ok=True)
        os.makedirs(f"{DEST_DIR}/labels/{split}", exist_ok=True)

    print(f"🧹 Filtering: Classes {KEEP_CLASSES}, Countries {KEEP_COUNTRIES}...")

    total_kept = 0
    splits_found = [s for s in ['train', 'val', 'test'] if os.path.exists(f"{SOURCE_ROOT}/{s}")]

    for split in splits_found:
        source_labels = f"{SOURCE_ROOT}/{split}/labels"
        source_images = f"{SOURCE_ROOT}/{split}/images"
        
        files = [f for f in os.listdir(source_labels) if f.endswith('.txt')]
        
        for file in tqdm(files, desc=f"Processing {split}"):
            # --- COUNTRY FILTER ---
            if not is_allowed_country(file):
                continue
            
            label_path = os.path.join(source_labels, file)
            
            # Read Labels
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            # --- CLASS FILTER ---
            new_lines = []
            has_target = False
            for line in lines:
                try:
                    cls = int(line.split()[0])
                    if cls in KEEP_CLASSES:
                        new_lines.append(line)
                        has_target = True
                except: pass
            
            if has_target:
                img_name = file.replace('.txt', '.jpg')
                img_path = os.path.join(source_images, img_name)
                
                if os.path.exists(img_path):
                    shutil.copy(img_path, f"{DEST_DIR}/images/{split}/{img_name}")
                    with open(f"{DEST_DIR}/labels/{split}/{file}", 'w') as out:
                        out.writelines(new_lines)
                    total_kept += 1

    print(f"✅ FILTERING COMPLETE! kept {total_kept} images (India/Japan/Czech + D00/D10)." )

## 4️⃣ Configure & Train

In [ ]:
# ===========================
# CREATE DATA.YAML
# ===========================
yaml_content = f"""
path: /content/rdd_cracks_only
train: images/train
val: images/val
names:
  0: Longitudinal_Crack
  1: Transverse_Crack
"""

with open("mini_rdd.yaml", "w") as f:
    f.write(yaml_content)
    
print("✅ mini_rdd.yaml created.")

# ===========================
# TRAIN YOLOv11
# ===========================
from ultralytics import YOLO

model_name = "yolo11m.pt"
print(f"🚀 Loading {model_name}...")
model = YOLO(model_name)

print(f"🎬 Starting Training (50 Epochs)...")
model.train(
    data="mini_rdd.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="pavi_cracks",
    name="dashboard_model"
)

print("✅ TRAINING COMPLETE!")

## 5️⃣ Download Model

In [ ]:
# ===========================
# PACKAGE & DOWNLOAD (FIXED PATH)
# ===========================
import shutil
from google.colab import files
import glob

# YOLO saves to runs/detect/{project}/{name}/weights/best.pt
search_paths = [
    "/content/runs/detect/pavi_cracks/**/weights/best.pt",
    "/content/runs/detect/**/weights/best.pt",
    "/content/pavi_cracks/**/weights/best.pt"
]

found = []
for pattern in search_paths:
    found.extend(glob.glob(pattern, recursive=True))

if found:
    best_path = sorted(found)[-1]
    print(f"✅ Found model at: {best_path}")
    shutil.copy(best_path, "/content/best_cracks_rdd.pt")
    print("⬇️ Downloading best_cracks_rdd.pt...")
    files.download("/content/best_cracks_rdd.pt")
else:
    print("❌ No best.pt found. Checking folder structure...")
    !ls -R /content/runs/ 2>/dev/null | head -30
    print("\nTry manually navigating to runs/detect/... to find best.pt")